<h1>SpaceX  Falcon 9 First Stage Landing Prediction</h1>


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


## Objectives

This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, we should be able to find some geographical patterns about launch sites.


In [2]:
!pip install wget

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9711 sha256=1044f3eadeaf6ba998063aaf751128de0432eda5f221e0303e59acfdd0f7af56
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\8a\b8\04\0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget


In [6]:
import folium
import wget
import pandas as pd

In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

<h2>Marking all launch sites on a map</h2>

In [8]:
spacex_df=pd.read_csv("spacex_launch_geo.csv")
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [9]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [10]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [11]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

Now, let's add a circle for each launch site in data frame `launch_sites`


Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


In [12]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for index, row in launch_sites_df.iterrows():
    
    coordinates = [row["Lat"], row["Long"]]
    label = row["Launch Site"]

    # Add circle
    folium.Circle(
        location=coordinates,
        radius=1000,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(label)).add_to(site_map)

    # Add marker (FIXED)
    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            html=f'<div style="font-size: 12; color:#d35400;"><b>{label}</b></div>'
        )
    ).add_to(site_map)

# SHOW MAP (outside loop)
site_map

<h2>Mark the success/failed launches for each site on the map</h2>

Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [14]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


In [15]:
marker_cluster = MarkerCluster()

In [18]:
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)

marker_cluster = MarkerCluster().add_to(site_map)

for index, row in spacex_df.iterrows():
    coordinates = [row["Lat"], row["Long"]]
    color = row["marker_color"]

    folium.Marker(
        location=coordinates,
        icon=folium.Icon(color=color)
    ).add_to(marker_cluster)

site_map

In [19]:

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [20]:
for index, row in spacex_df.iterrows():
    
    coordinates = [row["Lat"], row["Long"]]

    marker = folium.Marker(
        location=coordinates,
        icon=folium.Icon(
            color='white',
            icon_color=row["marker_color"]   # green/red
        )
    )
    
    marker_cluster.add_child(marker)


site_map

<h2>Calculate the distances between a launch site to its proximities</h2>

Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [21]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

You can calculate the distance between two points on the map based on their `Lat` and `Long`


In [22]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

Calculate the distance between the coastline point and the launch site.

In [23]:
launch_site_lat = 28.56367
launch_site_lon = -80.57163

coastline_lat = 28.56300
coastline_lon = -80.56700

distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon,
    coastline_lat, coastline_lon
)

distance_coastline
# Draw line
folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon],
               [coastline_lat, coastline_lon]],
    weight=2
).add_to(site_map)

# Add distance label
folium.Marker(
    [coastline_lat, coastline_lon],
    icon=folium.DivIcon(
        html=f'<div style="font-size: 12; color:red;"><b>{distance_coastline:.2f} KM</b></div>'
    )
).add_to(site_map)

site_map

In [24]:
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=folium.DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>'
    )
)

distance_marker.add_to(site_map)

 Draw a `PolyLine` between a launch site to the selected coastline point


In [25]:
coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]

folium.PolyLine(locations=coordinates, weight=2).add_to(site_map)

site_map

Launch site to the closest railway

In [26]:
rail_lat = 28.57200
rail_lon = -80.58500

distance_rail = calculate_distance(launch_site_lat, launch_site_lon, rail_lat, rail_lon)

# marker
folium.Marker(
    [rail_lat, rail_lon],
    icon=folium.DivIcon(
        html=f'<div style="font-size: 12; color:blue;"><b>{distance_rail:.2f} KM</b></div>'
    )
).add_to(site_map)

# line
folium.PolyLine([[launch_site_lat, launch_site_lon],[rail_lat, rail_lon]]).add_to(site_map)

Launch site to  the closest highway

In [27]:
highway_lat = 28.56350
highway_lon = -80.57000

distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)

folium.Marker(
    [highway_lat, highway_lon],
    icon=folium.DivIcon(
        html=f'<div style="font-size: 12; color:green;"><b>{distance_highway:.2f} KM</b></div>'
    )
).add_to(site_map)

folium.PolyLine([[launch_site_lat, launch_site_lon],[highway_lat, highway_lon]]).add_to(site_map)

Launchsite to the closest city

In [28]:
city_lat = 28.08360   # Melbourne
city_lon = -80.60810

distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)

folium.Marker(
    [city_lat, city_lon],
    icon=folium.DivIcon(
        html=f'<div style="font-size: 12; color:red;"><b>{distance_city:.2f} KM</b></div>'
    )
).add_to(site_map)

folium.PolyLine([[launch_site_lat, launch_site_lon],[city_lat, city_lon]]).add_to(site_map)

site_map

<ul>
<li>Railways are relatively close for transportation of materials</li>
<li>Highways are near for easy accessibility and logistics</li>
<li>They are very close to coastlines for safety</li>
<li> They are located away from cities to minimize risk to human  life</li>
</ul>


Launch sites are strategically located near coastlines for safety, allowing rockets to travel over water. They are also close to highways and railways to support transportation and logistics. However, they are positioned away from densely populated cities to reduce risk in case of failure.